In [ ]:
import os
import codecs
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# ==========================================
# 1. DOWNLOAD THE DATASET VIA KAGGLEHUB
# ==========================================
print("📥 Downloading dataset from Kaggle...")
# This pulls the popular movie genre classification benchmark dataset
path = kagglehub.dataset_download("hijest/genre-classification-dataset-imdb")
print(f"✅ Dataset downloaded to: {path}")

train_file_path = os.path.join(path, "Genre Classification Dataset", "train_data.txt")

# ==========================================
# 2. PARSE CUSTOM KAGGLE TXT FORMAT
# ==========================================
# The dataset uses ':::' as a delimiter rather than standard CSV commas
print("📖 Parsing dataset lines...")
train_data = []

with codecs.open(train_file_path, 'r', encoding='utf8') as f:
    for line in f:
        if line.strip():
            # Format: ID ::: TITLE ::: GENRE ::: DESCRIPTION
            parts = line.strip().split(" ::: ")
            if len(parts) == 4:
                train_data.append({
                    "id": parts[0],
                    "title": parts[1],
                    "genre": parts[2],
                    "description": parts[3]
                })

df = pd.DataFrame(train_data)
print(f"📊 Loaded {len(df)} movies.")
print(df[['title', 'genre']].head())

# ==========================================
# 3. TEXT PREPROCESSING
# ==========================================
print("🧹 Preprocessing movie descriptions...")
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_description(text):
    text = text.lower()
    # Simple alpha tokenization & stopword removal
    words = [word for word in text.split() if word.isalpha() and word not in stop_words]
    return " ".join(words)

# Taking a subset (e.g., 20,000 rows) just to optimize execution speed for testing
df_subset = df.sample(n=20000, random_state=42) if len(df) > 20000 else df
df_subset['clean_desc'] = df_subset['description'].apply(clean_description)

# ==========================================
# 4. TRAIN-TEST SPLIT & VECTORIZATION
# ==========================================
X = df_subset['clean_desc']
y = df_subset['genre']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("🧪 Extracting TF-IDF Features...")
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# ==========================================
# 5. MODEL TRAINING & EVALUATION
# ==========================================
print("🚀 Training Logistic Regression classifier...")
# Using balanced class weights because movie datasets are heavily skewed towards Drama/Comedy
model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train_tfidf, y_train)

# Predictions
y_pred = model.predict(X_test_tfidf)

print("\n🎯 --- EVALUATION PERFORMANCE ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# ==========================================
# 6. TEST ON A CUSTOM MOVIE PROMPT
# ==========================================
def predict_movie_genre(custom_plot):
    cleaned = clean_description(custom_plot)
    features = vectorizer.transform([cleaned])
    predicted_genre = model.predict(features)[0]
    return predicted_genre

custom_plot = "A group of rogue astronauts must travel into a massive wormhole to find a habitable planet and save humanity from an environmental collapse."
print("\n🔮 Testing custom plot context prediction:")
print(f"Plot: '{custom_plot}'")
print(f"Predicted Genre ➔ {predict_movie_genre(custom_plot).upper()}")

📥 Downloading dataset from Kaggle...


100%|██████████| 41.7M/41.7M [00:00<00:00, 68.2MB/s]

Extracting files...


✅ Dataset downloaded to: /root/.cache/kagglehub/datasets/hijest/genre-classification-dataset-imdb/versions/1
📖 Parsing dataset lines...
📊 Loaded 54214 movies.
                              title     genre
0      Oscar et la dame rose (2009)     drama
1                      Cupid (1997)  thriller
2  Young, Wild and Wonderful (1980)     adult
3             The Secret Sin (1915)     drama
4            The Unrecovered (2007)     drama
🧹 Preprocessing movie descriptions...


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


🧪 Extracting TF-IDF Features...
🚀 Training Logistic Regression classifier...

🎯 --- EVALUATION PERFORMANCE ---
Accuracy Score: 0.4515

Classification Report:

              precision    recall  f1-score   support

      action       0.26      0.44      0.33        90
       adult       0.17      0.31      0.22        35
   adventure       0.15      0.32      0.21        44
   animation       0.16      0.23      0.19        39
   biography       0.03      0.04      0.03        27
      comedy       0.57      0.42      0.48       553
       crime       0.14      0.28      0.19        43
 documentary       0.74      0.64      0.68       954
       drama       0.66      0.36      0.47      1015
      family       0.07      0.13      0.09        55
     fantasy       0.10      0.12      0.11        26
   game-show       0.57      0.67      0.62        12
     history       0.09      0.16      0.12        19
      horror       0.45      0.64      0.53       160
       music       0.30      0